# A strong triangular-hyperfield matroid violating ME1 and ME2

This notebook checks a counterexample in the paper "On the multiple exchange property of matroids over hyperfields" by Nathan Bowler, Changxin Ding, and Chayim Lowen. It checks **strong** matroids throughout, not just three-term Plücker relations. The counterexample was generated by ChatGPT; the assertions below independently verify its claims. (This notebook and its verification code were developed by Changxin Ding with assistance from ChatGPT.)

Choose **Run → Run All Cells**. The reusable functions live in the adjacent
`t1_checks.py`; the example is defined below. All computations here use exact
integers. No random search or numerical tolerance is used.

**Order:** reusable functions; definition of the example; strong-matroid check;
exhaustive ME1/ME2 checks; the two specific witnesses from the paper.


## 1. Reusable functions and the triangular hyperfield

In $T_1$, $a\boxplus b=[|a-b|,a+b]$ and $-a=a$. Thus exchange signs do not change
any value. For a finite list $t_1,\ldots,t_m\geq 0$, put $S=\sum_i t_i$ and
$M=\max_i t_i$. Its hypersum is the **closed interval**
$$\mathop{\boxplus}_i t_i=[\max(0,2M-S),S].$$
In particular, $0$ belongs to it exactly when $2M\leq S$.
The generic hypersum helper also handles the empty list, whose hypersum is $\{0\}$.
For an allowed exchange, the family of $I_2$ is always nonempty, even when
$I_1=\varnothing$.

The main functions are `check_strong_t1(E, r, chi)`, `check_ME1(E, r, chi)`,
and `check_ME2(E, r, chi)`. They return detailed dictionaries; read
`result['is_strong']` or `result['holds']`, **not** `bool(result)`.
The helper `exchange_witness` returns the data for a chosen triple.

Inputs: `E` is an ordered iterable of distinct hashable labels; `chi` is a
function on `frozenset` subsets, or a complete dictionary on all $r$-subsets.
Zero coordinates are allowed, so non-uniform matroids are supported.
Missing dictionary entries are errors. Integers, `Fraction`, `Decimal`, and
rational/decimal strings are accepted. Floating-point and symbolic irrational
inputs are deliberately rejected rather than given a misleading exact verdict.

The following cell imports the functions from [t1_checks.py](t1_checks.py).
Open that file in the JupyterLite file browser to read all definitions.
Future examples import the same file, avoiding duplicate implementations.
Strong-matroid validity is a separate check from the ME implications: verify it first.


In [1]:
from collections import Counter
from itertools import combinations
from t1_checks import (check_strong_t1, check_ME1, check_ME2,
                       exchange_witness, t1_hypersum_interval, t1_contains)


## 2. The example

Let $E=[8]$, $r=4$, $L=\{1,2,5,6\}$ and $H=\{3,4,5,6\}$, with complements in $E$.
Define
$$
\chi(B)=\begin{cases}
1& B=L\text{ or }B=L^c,\\
7& B=H\text{ or }B=H^c,\\
4& |B\cap L|=2\text{ and }|B\cap H|\in\{1,3\},\\
3&\text{otherwise}.
\end{cases}
$$
The first three cases are mutually exclusive, so their order does not matter
(the final case is the default). Indeed, for $B=L$ or $L^c$,
$|B\cap L|$ is $4$ or $0$; for $B=H$ or $H^c$, $|B\cap H|$ is $4$ or $0$.
Neither pair can satisfy the third case, and the four exceptional sets are distinct.
All 70 coordinates are positive.


In [2]:
E = tuple(range(1, 9))
r = 4
ground = frozenset(E)
L = frozenset({1, 2, 5, 6})
H = frozenset({3, 4, 5, 6})


def chi(B):
    B = frozenset(B)
    if len(B) != r or not B <= ground:
        raise ValueError("chi is defined on 4-subsets of E.")
    if B in (L, ground - L):
        return 1
    if B in (H, ground - H):
        return 7
    if len(B & L) == 2 and len(B & H) in {1, 3}:
        return 4
    return 3


coordinates = {frozenset(B): chi(B) for B in combinations(E, r)}
print("Coordinates:", len(coordinates))
print("Values and their multiplicities:", dict(sorted(Counter(coordinates.values()).items())))
assert len(coordinates) == 70 and all(x > 0 for x in coordinates.values())


Coordinates: 70
Values and their multiplicities: {1: 2, 3: 50, 4: 16, 7: 2}


## 3. Verify ALL strong Plücker relations

For every $I\in\binom{E}{r+1}$ and $J\in\binom{E}{r-1}$, the required relation is
$$
0\in\mathop{\boxplus}_{e\in I\setminus J}
\chi(I\setminus\{e\})\chi(J\cup\{e\}).
$$
Equivalently, twice the largest product must not exceed the sum of the products.
The omitted terms $e\in I\cap J$ vanish because of a repeated element.
Since $-1=1$ in $T_1$, the subset function extends symmetrically to distinct
ordered tuples and is zero on repeated tuples; this is its alternating extension.
Nonzeroness and **all** these relations give the strong Grassmann–Plücker axioms
(Baker–Bowler, Definition 3.11 and Theorem 3.13; reference below).

When $J\subseteq I$, write $I=J\sqcup\{a,b\}$. The two products are both
$$t=\chi(J\cup\{a\})\chi(J\cup\{b\}),$$
so the relation is $0\in t\boxplus t=[0,2t]$ and is automatic, including when
$t=0$. The checker therefore skips these pairs **before computing any products**.
This does not weaken the strong-matroid check: every remaining indexed relation
is still checked, not just the three-term relations.

Here there are $\binom85\binom83=3136$ indexed relations in total.
We skip $\binom85\binom53=560$ automatic two-term relations and explicitly
check the remaining **2576**. They need not be independent.
The report separates `relations_total`, `relations_skipped_trivial`, and
`relations_checked`. Its `by_terms` table contains only evaluated relations,
with $|I\setminus J|$ ranging from 3 to 5 in this example. Additional zero terms
from nonbases would still be counted. A nonnegative slack means the relation passes.


In [3]:
strong = check_strong_t1(E, r, chi)
print("Strong T1-matroid:", strong['is_strong'])
print("Support size:", strong['support_size'])
print("Total indexed relations:", strong['relations_total'])
print("Automatic relations skipped:", strong['relations_skipped_trivial'])
print("Relations checked explicitly:", strong['relations_checked'])
print("Violations:", strong['violations'])
print("\nTerms | Relations | Failures | Minimum (sum - 2*max)")
for k, item in strong['by_terms'].items():
    print(f"{k:5} | {item['checked']:9} | {item['failures']:8} | {item['minimum_slack']}")
assert strong['is_strong']
assert strong['relations_total'] == 3136
assert strong['relations_skipped_trivial'] == 560
assert strong['relations_checked'] == 2576
assert strong['violations'] == 0


Strong T1-matroid: True
Support size: 70
Total indexed relations: 3136
Automatic relations skipped: 560
Relations checked explicitly: 2576
Violations: 0

Terms | Relations | Failures | Minimum (sum - 2*max)
    3 |      1680 |        0 | 2
    4 |       840 |        0 | 2
    5 |        56 |        0 | 1


## 4. Exhaustive checks of ME1 and ME2

For each $B_1,B_2\in\binom{E}{r}$ and $I_1\subseteq B_1\setminus B_2$, write
$$a=\chi(B_1)\chi(B_2),\qquad
t_{I_2}=\chi(B_1-I_1+I_2)\chi(B_2+I_1-I_2),$$
where $I_2\subseteq B_2\setminus B_1$ and $|I_2|=|I_1|$.
ME1 tests the implication $a\geq\max t_{I_2}\Rightarrow
 a\in\mathop{\boxplus} t_{I_2}$. ME2 tests the implication
$a<\max t_{I_2}\Rightarrow a\in\mathop{\boxplus} t_{I_2}$.
Equality belongs to ME1. Membership is tested by the closed-interval formula.

The domain $I_1\subseteq B_1\setminus B_2$ is mandatory in both checkers.
The functions enumerate all ordered pairs of $r$-subsets, **including nonbases
and overlapping subsets**, and all allowed $I_1$, including $\varnothing$.
There is no requirement that $B_1$ and $B_2$ be disjoint. Since
$|B_1\setminus B_2|=|B_2\setminus B_1|$, every allowed $I_1$ has at least one
possible $I_2$. For $I_1=\varnothing$, the sole choice is $I_2=\varnothing$.
Both exchanged sets always have size $r$. The functions do not stop at the
first failure. The helper `exchange_witness` rejects any $I_1$ outside this domain.


In [4]:
reports = {}
print("Property | All triples | Hypothesis true | Violations")
for name, check in (("ME1", check_ME1), ("ME2", check_ME2)):
    result = check(E, r, chi)
    reports[name] = result
    print(f"{name:8} | {result['triples_checked']:11} | "
          f"{result['hypothesis_satisfied']:15} | {result['violations']}")
    assert not result['holds']
    assert result['triples_checked'] == 22470
    assert result['violations'] == 4
    assert result['violations_by_exchange_size'] == {2: 4}

assert reports['ME1']['hypothesis_satisfied'] == 15478
assert reports['ME2']['hypothesis_satisfied'] == 6992

# Each report contains the full first failure, including its exchange table:
# reports['ME1']['first_failure']


Property | All triples | Hypothesis true | Violations
ME1      |       22470 |           15478 | 4
ME2      |       22470 |            6992 | 4


## 5. The paper's ME1 witness

Take $B_1=H=\{3,4,5,6\}$, $B_2=H^c=\{1,2,7,8\}$, $I_1=\{5,6\}$.
The following table is computed from `chi`; its products are not hard-coded.


In [5]:
def show_witness(w):
    def fmt(B):
        return '{' + ','.join(map(str, B)) + '}'
    print("B1 =", fmt(w['B1']), " B2 =", fmt(w['B2']), " I1 =", fmt(w['I1']))
    print("I2      | B1-I1+I2    | B2+I1-I2    | product")
    for row in w['rows']:
        print(f"{fmt(row['I2']):7} | {fmt(row['left']):12} | {fmt(row['right']):12} | "
              f"{row['left_value']} * {row['right_value']} = {row['product']}")
    print("Target:", w['target'])
    print("Largest exchanged product:", w['max_product'])
    print("Hypersum interval:", w['hypersum_interval'])
    print("Target belongs to hypersum:", w['membership'])

w1 = exchange_witness(E, r, chi, {3,4,5,6}, {1,2,7,8}, {5,6})
show_witness(w1)
assert w1['products'] == (9, 9, 9, 9, 9, 1)
assert w1['target'] == 49 and w1['max_product'] == 9
assert w1['hypersum_interval'] == (0, 46)
assert w1['violates_ME1'] and not w1['violates_ME2']


B1 = {3,4,5,6}  B2 = {1,2,7,8}  I1 = {5,6}
I2      | B1-I1+I2    | B2+I1-I2    | product
{1,2}   | {1,2,3,4}    | {5,6,7,8}    | 3 * 3 = 9
{1,7}   | {1,3,4,7}    | {2,5,6,8}    | 3 * 3 = 9
{1,8}   | {1,3,4,8}    | {2,5,6,7}    | 3 * 3 = 9
{2,7}   | {2,3,4,7}    | {1,5,6,8}    | 3 * 3 = 9
{2,8}   | {2,3,4,8}    | {1,5,6,7}    | 3 * 3 = 9
{7,8}   | {3,4,7,8}    | {1,2,5,6}    | 1 * 1 = 1
Target: 49
Largest exchanged product: 9
Hypersum interval: (0, 46)
Target belongs to hypersum: False


Thus $49\geq9$, so the ME1 hypothesis holds, but
$$49\notin 9\boxplus9\boxplus9\boxplus9\boxplus9\boxplus1=[0,46].$$
The largest of the seven numbers is $49$, exceeding the sum $46$ of the other six.


## 6. The paper's ME2 witness

Take $B_1=\{1,2,3,4\}$, $B_2=\{5,6,7,8\}$, $I_1=\{1,2\}$.


In [6]:
w2 = exchange_witness(E, r, chi, {1,2,3,4}, {5,6,7,8}, {1,2})
show_witness(w2)
assert w2['products'] == (49, 9, 9, 9, 9, 1)
assert w2['target'] == 9 and w2['max_product'] == 49
assert w2['hypersum_interval'] == (12, 86)
assert w2['violates_ME2'] and not w2['violates_ME1']
print("\nAll example assertions passed.")


B1 = {1,2,3,4}  B2 = {5,6,7,8}  I1 = {1,2}
I2      | B1-I1+I2    | B2+I1-I2    | product
{5,6}   | {3,4,5,6}    | {1,2,7,8}    | 7 * 7 = 49
{5,7}   | {3,4,5,7}    | {1,2,6,8}    | 3 * 3 = 9
{5,8}   | {3,4,5,8}    | {1,2,6,7}    | 3 * 3 = 9
{6,7}   | {3,4,6,7}    | {1,2,5,8}    | 3 * 3 = 9
{6,8}   | {3,4,6,8}    | {1,2,5,7}    | 3 * 3 = 9
{7,8}   | {3,4,7,8}    | {1,2,5,6}    | 1 * 1 = 1
Target: 9
Largest exchanged product: 49
Hypersum interval: (12, 86)
Target belongs to hypersum: False

All example assertions passed.


Here $9<49$, so the ME2 hypothesis holds. The sum of the six exchanged products
is $86$ and their maximum is $49$, giving
$$49\boxplus9\boxplus9\boxplus9\boxplus9\boxplus1
=[2\cdot49-86,86]=[12,86].$$
The target $9$ lies below $12$. Equivalently, among the seven numbers
$9,49,9,9,9,9,1$, the largest is $49$ and the other six sum to $46$.


## 7. Reuse for another example

In this notebook the functions are already defined. In a new notebook in the
same directory, import them first:

```python
from t1_checks import check_strong_t1, check_ME1, check_ME2, exchange_witness
from fractions import Fraction

# Define E_new, r_new and a deterministic chi_new(B).
# Use Fraction(1, 3), not 1/3, for exact rational coordinates.
strong_new = check_strong_t1(E_new, r_new, chi_new)
assert strong_new['is_strong'], strong_new

me1_new = check_ME1(E_new, r_new, chi_new)
me2_new = check_ME2(E_new, r_new, chi_new)
print(me1_new['holds'], me2_new['holds'])
print(me1_new['first_failure'])
print(me2_new['first_failure'])
```

Both ME checkers always require $I_1\subseteq B_1\setminus B_2$.
These are finite exhaustive tests: the number of subsets grows quickly, so
large ground sets may be unsuitable for a browser.

### Mathematical reference
Matthew Baker and Nathan Bowler, *Matroids over hyperfields*,
[arXiv:1601.01204v5](https://arxiv.org/abs/1601.01204v5): triangle hyperfield on p. 2;
Definition 3.11 and Theorem 3.13 on p. 14. The hypersum interval formula above
follows by iterating $a\boxplus b=[|a-b|,a+b]$.
